# Module 2: Classical AI - Adversarial Search (Minimax)

Adversarial search is used in multi-agent environments where agents have conflicting goals (e.g., chess, checkers, tic-tac-toe).

In this notebook, we will explore:
1. **The Minimax Algorithm**: Backtracking decision-making to find the optimal move assuming an opponent plays optimally.
2. **Alpha-Beta Pruning**: An optimization that decreases the number of nodes evaluated by the Minimax algorithm in its search tree.
3. **Interactive Tic-Tac-Toe**: Playing against an unbeatable AI agent.

In [ ]:
import math
import random

---

## Tic-Tac-Toe Game Engine

Let's represent a Tic-Tac-Toe board as a list of 9 elements, where:
- `' '` (space) represents an empty cell.
- `'X'` represents Player X.
- `'O'` represents Player O.

In [ ]:
def print_board(board):
    for i in range(3):
        row = board[i*3 : (i+1)*3]
        print(" | ".join(row))
        if i < 2:
            print("---------")

def get_empty_cells(board):
    return [i for i, cell in enumerate(board) if cell == ' ']

def check_winner(board, player):
    # Check rows, columns, and diagonals
    win_conditions = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8], # rows
        [0, 3, 6], [1, 4, 7], [2, 5, 8], # cols
        [0, 4, 8], [2, 4, 6]             # diagonals
    ]
    return any(all(board[cell] == player for cell in condition) for condition in win_conditions)

---

## 1. Minimax Algorithm

Minimax simulates all possible future moves. The Maximizing player tries to get the highest score, while the Minimizing player tries to get the lowest score.

- Utility: $+10$ if AI wins, $-10$ if Human wins, $0$ if Draw.

In [ ]:
def evaluate(board, ai_player, human_player):
    if check_winner(board, ai_player):
        return 10
    elif check_winner(board, human_player):
        return -10
    return 0

def minimax(board, depth, is_maximizing, ai_player, human_player):
    score = evaluate(board, ai_player, human_player)
    
    if score == 10 or score == -10:
        return score
        
    empty_cells = get_empty_cells(board)
    if not empty_cells:
        return 0 # Draw
        
    if is_maximizing:
        best_val = -math.inf
        for cell in empty_cells:
            board[cell] = ai_player
            value = minimax(board, depth + 1, False, ai_player, human_player)
            board[cell] = ' '
            best_val = max(best_val, value)
        return best_val
    else:
        best_val = math.inf
        for cell in empty_cells:
            board[cell] = human_player
            value = minimax(board, depth + 1, True, ai_player, human_player)
            board[cell] = ' '
            best_val = min(best_val, value)
        return best_val

---

## 2. Alpha-Beta Pruning

Alpha-Beta pruning passes two parameters through recursion:
- $\alpha$: Best score Maximizer is assured of.
- $\beta$: Best score Minimizer is assured of.

If $\beta \le \alpha$, we prune the remaining branches.

In [ ]:
def minimax_alphabeta(board, depth, alpha, beta, is_maximizing, ai_player, human_player):
    score = evaluate(board, ai_player, human_player)
    if score == 10 or score == -10:
        return score
        
    empty_cells = get_empty_cells(board)
    if not empty_cells:
        return 0
        
    if is_maximizing:
        best_val = -math.inf
        for cell in empty_cells:
            board[cell] = ai_player
            value = minimax_alphabeta(board, depth + 1, alpha, beta, False, ai_player, human_player)
            board[cell] = ' '
            best_val = max(best_val, value)
            alpha = max(alpha, best_val)
            if beta <= alpha:
                break # Prune branch
        return best_val
    else:
        best_val = math.inf
        for cell in empty_cells:
            board[cell] = human_player
            value = minimax_alphabeta(board, depth + 1, alpha, beta, True, ai_player, human_player)
            board[cell] = ' '
            best_val = min(best_val, value)
            beta = min(beta, best_val)
            if beta <= alpha:
                break # Prune branch
        return best_val

def find_best_move(board, ai_player, human_player):
    best_val = -math.inf
    best_move = -1
    for cell in get_empty_cells(board):
        board[cell] = ai_player
        # Use Alpha-Beta for performance optimization
        move_val = minimax_alphabeta(board, 0, -math.inf, math.inf, False, ai_player, human_player)
        board[cell] = ' '
        if move_val > best_val:
            best_val = move_val
            best_move = cell
    return best_move

---

## 3. Play Against the AI

Run the following code block to play a game of Tic-Tac-Toe against the AI. 
Since the AI plays optimally using Minimax, you will **not** be able to beat it! Best outcome is a draw.

In [ ]:
def play_game():
    board = [' '] * 9
    human = 'X'
    ai = 'O'
    
    print("Welcome to Tic-Tac-Toe against the Minimax AI!")
    print("Cells are numbered 0-8 from top-left to bottom-right:")
    print("0 | 1 | 2\n---------\n3 | 4 | 5\n---------\n6 | 7 | 8\n")
    
    # Randomly decide who goes first
    turn = 'Human' if random.random() < 0.5 else 'AI'
    print(f"{turn} goes first!\n")
    
    while get_empty_cells(board):
        if turn == 'Human':
            print_board(board)
            move = -1
            while move not in get_empty_cells(board):
                try:
                    move = int(input("Enter your move (0-8): "))
                except ValueError:
                    print("Please enter a valid integer.")
            board[move] = human
            if check_winner(board, human):
                print_board(board)
                print("Congratulations! You beat the AI!") # Note: Under minimax, this won't happen
                return
            turn = 'AI'
        else:
            print("\nAI is calculating the best move...")
            move = find_best_move(board, ai, human)
            board[move] = ai
            if check_winner(board, ai):
                print_board(board)
                print("AI wins! Better luck next time.")
                return
            turn = 'Human'
            
    print_board(board)
    print("It's a draw!")

# Uncomment and run this in your local Jupyter console or notebook interface to play!
# play_game()

### 🏋️ Exercise: Depth Penalty

Right now, the AI doesn't care if it wins on the next move or in 5 moves; all winning branches evaluate to $+10$.

Modify the evaluation to include a depth penalty so the AI values **faster wins** over slower ones:
- AI win score: $10 - \text{depth}$
- Human win score: $-10 + \text{depth}$

In [ ]:
# Implement depth penalty in minimax here